<a href="https://colab.research.google.com/github/he-man-david/small-transformer-model-arithmetic-math/blob/master/create-dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Generator that we use to create training data

Training data will be strings like "100 + 9 / 3", and the label will be "103".
We will generate all our training data programmatically. To keep model small, we will keep length of expression short
and only float up to 2 decimals.

Example:
```
[
    [
        "-51.69 / -59.25 - -98.82", <-- math expression/instruction
        "99.69" <-- our label/result
    ],
    [
        "62.89 / 50.31",
        "1.25"
    ],
    [
        "28.72 * 70.67 * -99.85 + 54.13 + 43.59",
        "-202562.07"
    ]
]
```

In [1]:
import random
import json
import os

def generate_arithmetic_dataset(
    size: int = 10000,
    min_num_of_ops: int = 1,
    max_num_of_ops: int = 2, 
    operators: list[str] = ['+', '-', '*', '/'], 
    operator_weights: list[float] = None,
    num_type: str = "int",  # Options: 'int', 'float', 'mixed'
):
    dataset = []
    
    # Validation check to catch spelling mistakes early
    if num_type not in ["int", "float", "mixed"]:
        raise ValueError("num_type must be either 'int', 'float', or 'mixed'")
    
    for i in range(size):
        expression_arr = []
        
        rand_ops_len = random.randint(min_num_of_ops, max_num_of_ops) * 2 + 1
            
        for j in range(rand_ops_len):
            if (j % 2) == 0: # Even idx for numbers
                
                # Determine what type of number to generate based on your config
                current_type = num_type
                if num_type == "mixed":
                    current_type = random.choice(["int", "float"])
                
                # Generate the base number
                if current_type == "int":
                    random_num = random.randint(-100, 100)
                else:  # float
                    random_num = round(random.uniform(-100.0, 100.0), 2)
                
                # Prevent Division by Zero (Handles both 0 and 0.0)
                if j > 0 and expression_arr[j-1] == '/' and float(random_num) == 0.0:
                    while float(random_num) == 0.0:
                        if current_type == "int":
                            random_num = random.randint(-100, 100)
                        else:
                            random_num = round(random.uniform(-100.0, 100.0), 2)
                
                expression_arr.append(str(random_num))
            else: # Odd idx for operators
                if operator_weights is not None:
                    random_operator = random.choices(operators, weights=operator_weights, k=1)[0]
                else:
                    random_operator = random.choice(operators)
                expression_arr.append(random_operator)
        
        expression_str = " ".join(expression_arr)
        
        try:
            raw_result = eval(expression_str)
            
            if num_type == "int" and '/' not in operators:
                result = str(int(raw_result))
            else:
                result = str(round(raw_result, 2))
                
            dataset.append([expression_str, result])
        except ZeroDivisionError:
            # Fallback safety buffer just in case eval hits a weird precision edge case
            continue

    return dataset

## Making the training data simpler for model

Training with + - * / operators and long sequence was impossible for transformer model to learn anything, regardless of d_model or epochs. Problem isn't in what the activation or layer units or how deep the FFN is ...etc I must fundamentally rethink the data I use to train.

### Start with addition (like in elementary school!)
- Integer only, [+], 1 to 2 operations, size = 50,000
- Integer only, [+], 1 to 5 operations, size = 100,000
- Integer + float mixed, [+], 1 to 2 operations, size = 200,000
- Integer + float mixed, [+], 1 to 5 operations, size = 300,000

### Move on to subtraction
- Integer + float mixed, [+, -] with weight [40, 60], 1 to 2 operations, size = 300,000 
- Integer + float mixed, [+, -] with weight [10, 90], 1 to 5 operations, size = 300,000

### Move on to multiplication
- Integer + float mixed, [+, -, *] with weight [45, 45, 10], 1 to 2 operations, size = 300,000 <
- Integer + float mixed, [+, -, *] with weight [33, 33, 34], 1 to 5 operations, size = 300,000

### Move on to divide
- Integer + float mixed, [+, -, *, /] with weight [32, 32, 32, 4], 1 to 2 operations, size = 300,000
- Integer + float mixed, [+, -, *, /] with weight [30, 30, 30, 10], 1 to 2 operations, size = 300,000

### All together
- Integer + float mixed, [+, -, *, /] with weight [25, 25, 25, 25], 1 to 2 operations, size = 300,000
- Integer + float mixed, [+, -, *, /] with weight [25, 25, 25, 25], 1 to 5 operations, size = 300,000

In [4]:
num_type = 'mixed'
size = 300000
min_num_of_ops = 1
max_num_of_ops = 2
operator_weights = [45, 45, 10]
operators = ['+', '-', '*']

dataset_name = '6_mixed_plus_sub_mult_1_to_2_size_300k'

dataset = generate_arithmetic_dataset(size=size, min_num_of_ops=min_num_of_ops, max_num_of_ops=max_num_of_ops, operators=operators, operator_weights=operator_weights, num_type=num_type)

print(json.dumps(dataset[:3], indent=4))
print(f"completed dataset creation of size {size}....")

[
    [
        "38.01 + 96.78",
        "134.79"
    ],
    [
        "32.44 + 68 - 99",
        "1.44"
    ],
    [
        "-42.11 + 24",
        "-18.11"
    ]
]
completed dataset creation of size 300000....


In [5]:
# Saving dataset to dir
repo_dir = "/teamspace/studios/this_studio/small-transformer-model-arithmetic-math/data"
file_path = os.path.join(repo_dir, f"{dataset_name}.json")
folder_path = os.path.dirname(file_path)

if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Created missing directory: {folder_path}")
    
with open(file_path, 'w') as f:
    json.dump(dataset, f, indent=4)
    print(f"Completed creating in {file_path}")

Completed creating in /teamspace/studios/this_studio/small-transformer-model-arithmetic-math/data/6_mixed_plus_sub_mult_1_to_2_size_300k.json
